In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, confusion_matrix, classification_report
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

load data and assign train test

In [2]:
df1=pd.read_csv('train.csv')
df2=pd.read_csv('test.csv')
X_train=df1[['Name','Pclass','Age','Sex','Parch','SibSp','Fare']]
y_train=df1['Survived']
X_test=df2[['Name','Pclass','Age','Sex','Parch','SibSp','Fare']]

In [3]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    891 non-null    object 
 1   Pclass  891 non-null    int64  
 2   Age     714 non-null    float64
 3   Sex     891 non-null    object 
 4   Parch   891 non-null    int64  
 5   SibSp   891 non-null    int64  
 6   Fare    891 non-null    float64
dtypes: float64(2), int64(3), object(2)
memory usage: 48.9+ KB


data preprocessing (handeling missing values)

In [4]:
X_train.isna().sum().sort_values(ascending=False)

Age       177
Name        0
Pclass      0
Sex         0
Parch       0
SibSp       0
Fare        0
dtype: int64

In [5]:
imp_age_X_train=SimpleImputer(strategy='mean')
X_train['Age']=imp_age_X_train.fit_transform(X_train[['Age']])

C:\Users\hp\AppData\Local\Temp\ipykernel_21084\4144384455.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Age']=imp_age_X_train.fit_transform(X_train[['Age']])


In [6]:
X_train.isna().sum()

Name      0
Pclass    0
Age       0
Sex       0
Parch     0
SibSp     0
Fare      0
dtype: int64

In [7]:
y_train.isna().sum()

np.int64(0)

In [8]:
print(X_test.info())
X_test.isna().sum().sort_values(ascending=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    418 non-null    object 
 1   Pclass  418 non-null    int64  
 2   Age     332 non-null    float64
 3   Sex     418 non-null    object 
 4   Parch   418 non-null    int64  
 5   SibSp   418 non-null    int64  
 6   Fare    417 non-null    float64
dtypes: float64(2), int64(3), object(2)
memory usage: 23.0+ KB
None


Age       86
Fare       1
Name       0
Pclass     0
Sex        0
Parch      0
SibSp      0
dtype: int64

In [9]:
imp_age_X_test=SimpleImputer(strategy='mean')
X_test['Age']=imp_age_X_test.fit_transform(X_test[['Age']])
X_test=X_test.dropna(subset=['Fare'])

C:\Users\hp\AppData\Local\Temp\ipykernel_21084\1950311554.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test['Age']=imp_age_X_test.fit_transform(X_test[['Age']])


In [10]:
X_test.isna().sum().sort_values(ascending=False)

Name      0
Pclass    0
Age       0
Sex       0
Parch     0
SibSp     0
Fare      0
dtype: int64

Converting categorical data to numerical data

In [11]:
X_train=X_train.drop('Sex',axis=1).join(pd.get_dummies(X_train['Sex'],drop_first=True).astype(int))
X_test=X_test.drop('Sex',axis=1).join(pd.get_dummies(X_test['Sex'],drop_first=True).astype(int))
print(X_train.head())
print(X_test.head())

                                                Name  Pclass   Age  Parch  \
0                            Braund, Mr. Owen Harris       3  22.0      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...       1  38.0      0   
2                             Heikkinen, Miss. Laina       3  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)       1  35.0      0   
4                           Allen, Mr. William Henry       3  35.0      0   

   SibSp     Fare  male  
0      1   7.2500     1  
1      1  71.2833     0  
2      0   7.9250     0  
3      1  53.1000     0  
4      0   8.0500     1  
                                           Name  Pclass   Age  Parch  SibSp  \
0                              Kelly, Mr. James       3  34.5      0      0   
1              Wilkes, Mrs. James (Ellen Needs)       3  47.0      0      1   
2                     Myles, Mr. Thomas Francis       2  62.0      0      0   
3                              Wirz, Mr. Albert       3  27.0    

In [12]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    891 non-null    object 
 1   Pclass  891 non-null    int64  
 2   Age     891 non-null    float64
 3   Parch   891 non-null    int64  
 4   SibSp   891 non-null    int64  
 5   Fare    891 non-null    float64
 6   male    891 non-null    int64  
dtypes: float64(2), int64(4), object(1)
memory usage: 48.9+ KB


KNeighborsClassifier

In [13]:
X_Train=X_train.drop('Name',axis=1)
X_Test=X_test.drop('Name',axis=1)
knn=KNeighborsClassifier(5)
knn.fit(X_Train,y_train)
y_pred=knn.predict(X_Test)
X_test['Predicted_Survived']=y_pred
predicted_survivors = X_test[X_test['Predicted_Survived'] == 1]['Name']
print(predicted_survivors.tolist())
print(f"{predicted_survivors.count()} passengers are predicted to survive by K-Nearest Neighbors.")


['Wirz, Mr. Albert', 'Connolly, Miss. Kate', 'Caldwell, Mr. Albert Francis', 'Davies, Mr. John Samuel', 'Snyder, Mrs. John Pillsbury (Nelle Stevenson)', 'Chaffee, Mrs. Herbert Fuller (Carrie Constance Toogood)', 'del Carlo, Mrs. Sebastiano (Argenia Genovesi)', 'Ilmakangas, Miss. Ida Livija', 'Olsen, Master. Artur Karl', 'Flegenheim, Mrs. Alfred (Antoinette)', 'Williams, Mr. Richard Norris II', 'Ryerson, Mrs. Arthur Larned (Emily Maria Borie)', 'Ostby, Miss. Helene Ragnhild', 'Samaan, Mr. Elias', 'Louch, Mr. Charles Alexander', 'Jefferys, Mr. Clifford Thomas', 'Mock, Mr. Philipp Edmund', 'Hee, Mr. Ling', 'Corbett, Mrs. Walter H (Irene Colvin)', 'Kimball, Mrs. Edwin Nelson Jr (Gertrude Parsons)', 'Bucknell, Mrs. William Robert (Emma Eliza Ward)', 'Coutts, Mrs. William (Winnie Minnie" Treanor)"', 'Smith, Mr. Lucien Philip', 'Hocking, Miss. Ellen Nellie""', 'Fortune, Miss. Ethel Flora', 'Chaudanson, Miss. Victorine', 'McCrae, Mr. Arthur Gordon', 'Bradley, Miss. Bridget Delia', 'Ryerson, Ma

LogesticRegression

In [14]:
lr=LogisticRegression()
lr.fit(X_Train,y_train)

LogisticRegression()

In [15]:
y_train_pred = lr.predict(X_Test)
print(y_train_pred)
X_test['Predicted_Survived_by_lr']=y_train_pred
predicted_survivors_by_lr = X_test[X_test['Predicted_Survived_by_lr'] == 1]['Name']
print(predicted_survivors_by_lr.tolist())
print(f"{predicted_survivors_by_lr.count()} passengers are predicted to survive by Logistic Regression.")

[0 0 0 0 1 0 1 0 1 0 0 0 1 0 1 1 0 0 1 0 0 0 1 1 1 0 1 0 0 0 0 0 0 0 0 0 1
 1 0 0 0 0 0 1 1 0 0 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 0 1 1 1 1 0 0 1 1 0 1 1
 1 1 0 1 0 1 0 0 0 0 0 0 1 1 1 0 1 0 1 0 1 0 1 0 1 0 1 0 0 0 1 0 0 0 0 0 0
 1 1 1 1 0 0 1 0 1 1 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0
 0 0 1 0 1 0 0 1 1 0 1 1 0 1 0 0 1 0 0 1 1 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1 0
 1 0 0 0 0 0 0 0 0 0 1 1 0 1 1 0 0 1 0 0 1 0 1 0 0 0 0 1 0 0 1 0 1 0 1 0 1
 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 0 1 1 1 0 1 0 0 0 0 0 1 0
 0 0 1 1 0 0 0 0 1 0 0 0 1 1 0 1 0 0 0 0 1 0 1 1 1 0 0 1 0 0 1 1 0 0 0 0 1
 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 1
 0 0 0 0 0 0 0 0 0 1 0 1 0 1 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 0 1 0 0 1 1 0 0
 1 0 0 1 1 1 0 0 0 0 0 1 1 0 1 0 0 0 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 0 0 0 0
 1 1 1 1 1 0 1 0 0 0]
['Hirvonen, Mrs. Alexander (Helga E Lindqvist)', 'Connolly, Miss. Kate', 'Abrahim, Mrs. Joseph (Sophie Halaut Easu)', 'Snyder, Mrs. John Pillsbury (Nelle

Comparing between both the models as we dont have any y_test label we can see the prediction of number of passengers survived by K-Nearest Neighbors are 154 and Logistic Regression are 155